# 02 - 2-Step RAG Implementation

## 🎯 Learning Objectives

By the end of this notebook, you will:
- Understand the 2-Step RAG architecture
- Build a RAG chain with LangChain
- Create a Q&A system grounded in retrieved knowledge
- Compare responses with and without RAG
- Understand when to use 2-Step RAG vs other architectures

## 📚 What is 2-Step RAG?

In **2-Step RAG**, retrieval always happens before generation:

```plain
User Question → Retrieve Documents → Generate Answer → Return to User
```

### Characteristics:

| Feature | Description |
|---------|-------------|
| **Control** | ✅ High - predictable execution flow |
| **Flexibility** | ❌ Low - always retrieves before answering |
| **Latency** | ⚡ Fast - known number of LLM calls (usually 1) |
| **Use Cases** | FAQs, documentation bots, simple Q&A |

### When to use 2-Step RAG:
- You always need external knowledge to answer
- Predictable latency is important
- Simple, transparent system behavior
- Questions are well-scoped

## 🔧 Setup

Let's set up our environment and reload the knowledge base from Notebook 01.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv("../.env")

print("✅ Environment loaded")

✅ Environment loaded


In [2]:
from langchain_dev_utils.chat_models import register_model_provider, load_chat_model
from langchain_dev_utils.embeddings import register_embeddings_provider, load_embeddings

# SiliconFlow configuration
SILICONFLOW_BASE_URL = os.getenv("SILICONFLOW_BASE_URL", "https://api.siliconflow.cn/v1")

# Register providers
register_model_provider(
    provider_name="siliconflow",
    chat_model="openai-compatible",
    base_url=SILICONFLOW_BASE_URL,
)

register_embeddings_provider(
    provider_name="siliconflow",  # Fixed: use provider_name instead of provider
    embeddings_model="openai-compatible",
    base_url=SILICONFLOW_BASE_URL,
)

# Load models
CHAT_MODEL_NAME = os.getenv("SILICONFLOW_CHAT_MODEL", "Qwen/Qwen2.5-7B-Instruct")
EMBEDDING_MODEL_NAME = os.getenv("SILICONFLOW_EMBEDDING_MODEL", "BAAI/bge-m3")

chat_model = load_chat_model(f"siliconflow:{CHAT_MODEL_NAME}")
embeddings = load_embeddings(f"siliconflow:{EMBEDDING_MODEL_NAME}")

print(f"✅ Chat Model: {CHAT_MODEL_NAME}")
print(f"✅ Embedding Model: {EMBEDDING_MODEL_NAME}")

✅ Chat Model: Qwen/Qwen2.5-7B-Instruct
✅ Embedding Model: BAAI/bge-m3


In [ ]:
from langchain_oceanbase.vectorstores import OceanbaseVectorStore

# OceanBase connection
connection_args = {
    "host": os.getenv("OCEANBASE_HOST", "127.0.0.1"),
    "port": int(os.getenv("OCEANBASE_PORT", "2881")),
    "user": os.getenv("OCEANBASE_USER", "root@test"),
    "password": os.getenv("OCEANBASE_PASSWORD", ""),
    "db_name": os.getenv("OCEANBASE_DB", "test"),
}

# Load existing vector store from Notebook 01
vector_store = OceanbaseVectorStore(
    embedding_function=embeddings,
    table_name="langchain_knowledge_base",
    connection_args=connection_args,
    vidx_metric_type="cosine",
    drop_old=False,  # Use existing data
)

print("✅ Connected to existing knowledge base")

# Create retriever
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Retrieve top 3 most relevant documents
)

print("✅ Retriever created (k=3)")

## 🧪 Baseline: LLM Without RAG

First, let's see what the LLM knows without retrieval.

In [ ]:
from langchain.schema import HumanMessage

question = "What were Nike's total revenues and key financial highlights in fiscal 2023?"

# Ask LLM directly without retrieval
response_without_rag = chat_model.invoke([HumanMessage(content=question)])

print("❓ Question (without RAG):")
print(f"   {question}")
print(f"\n💭 LLM Response (no retrieval):")
print(f"   {response_without_rag.content}")

## 🔗 Step 1: Simple RAG Chain

Let's build a basic RAG chain that retrieves documents and passes them to the LLM.

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough

# Define RAG prompt template
rag_template = """You are a helpful assistant answering questions about Nike's business based on their 10-K annual report.

Use the following retrieved context to answer the question. If you don't know the answer based on the context, say so.

Context:
{context}

Question: {question}

Answer:"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)

# Helper function to format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | chat_model
    | StrOutputParser()
)

print("✅ RAG chain created")
print("\n🔗 Chain structure:")
print("   Question → Retrieve Docs → Format Context → LLM → Parse Output")

## 🎯 Step 2: Test RAG Chain

Now let's use the RAG chain with the same question.

In [6]:
# Use RAG chain
response_with_rag = rag_chain.invoke(question)

print("❓ Question (with RAG):")
print(f"   {question}")
print(f"\n✅ RAG Response:")
print(f"   {response_with_rag}")

# Show retrieved context
print("\n📚 Retrieved Context:")
retrieved_docs = retriever.invoke(question)
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n   Document {i}:")
    print(f"   {doc.page_content[:150].replace(chr(10), ' ')}...")
    print(f"   Page: {doc.metadata.get('page', 'N/A')}")

❓ Question (with RAG):
   What were Nike's total revenues and key financial highlights in fiscal 2023?

✅ RAG Response:
   In fiscal 2023, Nike's total revenues were $51.2 billion. Key financial highlights for the fiscal year include:

- Revenues increased by 10% on a reported basis and by 16% on a currency-neutral basis compared to fiscal 2022.
- NIKE Direct revenues grew by 14%, reaching $21.3 billion, and accounted for approximately 44% of total NIKE Brand revenues.
- The gross margin decreased by 250 basis points to 43.5%, primarily due to higher product costs, higher markdowns, and unfavorable changes in foreign currency exchange rates, partly offset by strategic pricing actions.
- Inventories were $8.5 billion, unchanged from the prior year, thanks to inventory management actions throughout fiscal 2023.
- Nike returned $7.5 billion to shareholders via share repurchases and dividends.

📚 Retrieved Context:

   Document 1:
   speed and responsiveness as we serve consumers globally.

## 📊 Compare Responses

In [7]:
print("="*80)
print("COMPARISON: RAG vs No RAG")
print("="*80)

print(f"\n❓ Question:\n   {question}")

print(f"\n{'─'*80}")
print("🔴 WITHOUT RAG (LLM knowledge only):")
print(f"{'─'*80}")
print(f"{response_without_rag.content}")

print(f"\n{'─'*80}")
print("🟢 WITH RAG (Retrieved knowledge):")
print(f"{'─'*80}")
print(f"{response_with_rag}")

print(f"\n{'='*80}")
print("\n💡 Notice: RAG provides more specific, grounded answers based on retrieved context!")

COMPARISON: RAG vs No RAG

❓ Question:
   What were Nike's total revenues and key financial highlights in fiscal 2023?

────────────────────────────────────────────────────────────────────────────────
🔴 WITHOUT RAG (LLM knowledge only):
────────────────────────────────────────────────────────────────────────────────
Nike reported its fiscal year 2023 (FY23) results on May 23, 2023. Here are the key financial highlights:

- **Total Revenues**: Nike's total revenue for fiscal year 2023 was approximately $44.3 billion, representing year-over-year growth of 13% compared to $39.1 billion in fiscal year 2022.

- **Net Income**: Nike reported a net income of $5.67 billion for fiscal year 2023, up from $4.89 billion in fiscal year 2022, showcasing a 16% increase.

- **Earnings Per Share (EPS)**: Nike's EPS for fiscal year 2023 was $2.77, up from $2.35 in the previous year, indicating a 18% increase.

- **Digital Revenue**: Digital sales continued to grow significantly, accounting for over 30% 

## 🚀 Step 3: Interactive Q&A System

Let's test the RAG system with multiple questions.

In [8]:
# Test questions about Nike
test_questions = [
    "What are Nike's main business segments?",
    "What were Nike's total revenues in fiscal 2023?",
    "What are the key risk factors facing Nike?",
    "What is Nike's growth strategy?",
]

print("🎯 Testing RAG System with Multiple Nike Questions")
print("="*80)

for i, q in enumerate(test_questions, 1):
    print(f"\n{'─'*80}")
    print(f"Question {i}: {q}")
    print(f"{'─'*80}")
    
    # Get answer
    answer = rag_chain.invoke(q)
    print(f"\n💬 Answer:\n{answer}")
    
    # Show top retrieved doc
    docs = retriever.invoke(q)
    print(f"\n📄 Top Retrieved Source: Page {docs[0].metadata.get('page', 'N/A')}")

print(f"\n{'='*80}")

🎯 Testing RAG System with Multiple Nike Questions

────────────────────────────────────────────────────────────────────────────────
Question 1: What are Nike's main business segments?
────────────────────────────────────────────────────────────────────────────────

💬 Answer:
Nike's main business segments are described as follows:

1. **North America**: This segment includes the sales and operating results for the North American region, covering both NIKE and Jordan brands.
2. **Europe, Middle East & Africa (EMEA)**: This segment covers the sales and operating results for the regions of Europe, the Middle East, and Africa, again including both NIKE and Jordan brands.
3. **Greater China**: This segment focuses on the sales and operating results for the Greater China region, encompassing both NIKE and Jordan brands.
4. **Asia Pacific & Latin America (APLA)**: This segment includes the sales and operating results for the Asia Pacific and Latin American regions, covering both NIKE and Jorda

## 🔍 Step 4: RAG with Source Citations

Let's enhance our RAG to include source citations.

In [ ]:
# Enhanced RAG prompt with citations
citation_template = """You are a helpful assistant answering questions about Nike's business based on their 10-K annual report.

Use the following retrieved context to answer the question. Include references to which source(s) you used (by page number).
If you don't know the answer based on the context, say so.

Context:
{context}

Question: {question}

Provide your answer and mention which sources (pages) you referenced.

Answer:"""

citation_prompt = ChatPromptTemplate.from_template(citation_template)

# Enhanced format function with source info
def format_docs_with_sources(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get('page', 'Unknown')
        formatted.append(f"[Source {i} - Page {page}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# Build enhanced chain
citation_rag_chain = (
    {"context": retriever | format_docs_with_sources, "question": RunnablePassthrough()}
    | citation_prompt
    | chat_model
    | StrOutputParser()
)

print("✅ Enhanced RAG chain created with source citations")

In [10]:
# Test with citations
question = "What are Nike's key competitive strengths and market position?"

answer_with_citations = citation_rag_chain.invoke(question)

print(f"❓ Question: {question}")
print(f"\n💬 Answer with Citations:\n{answer_with_citations}")

❓ Question: What are Nike's key competitive strengths and market position?

💬 Answer with Citations:
Nike's key competitive strengths and market position are rooted in its strong brand presence, innovation, and global reach. These strengths are evident from the following aspects:

1. **Global Leadership**: As the largest seller of athletic footwear and apparel in the world, Nike benefits from a significant market share, which helps in leveraging economies of scale and maintaining a strong brand presence.

2. **Brand Portfolio**: Nike's diverse brand portfolio, including the NIKE Brand, Jordan Brand, and Converse, caters to various segments of the market. This diversity not only increases its market reach but also provides resilience against shifts in consumer preferences or market trends.

3. **Innovation and Product Quality**: The company emphasizes product attributes such as quality, performance, and reliability, as well as new product innovations and development. This focus on quali

## ⚡ Step 5: Performance Analysis

Let's measure the performance characteristics of our 2-Step RAG system.

In [11]:
import time

def measure_rag_performance(question):
    """Measure retrieval and generation times"""
    
    # Measure retrieval time
    start_retrieval = time.time()
    retrieved_docs = retriever.invoke(question)
    retrieval_time = time.time() - start_retrieval
    
    # Measure total RAG time
    start_total = time.time()
    answer = rag_chain.invoke(question)
    total_time = time.time() - start_total
    
    # Calculate generation time
    generation_time = total_time - retrieval_time
    
    return {
        "question": question,
        "answer": answer,
        "num_docs_retrieved": len(retrieved_docs),
        "retrieval_time": retrieval_time,
        "generation_time": generation_time,
        "total_time": total_time,
    }

# Test performance with Nike question
test_q = "What are Nike's digital transformation initiatives and e-commerce strategy?"
perf = measure_rag_performance(test_q)

print("⚡ 2-Step RAG Performance Metrics")
print("="*80)
print(f"\n❓ Question: {perf['question']}")
print(f"\n📊 Performance:")
print(f"   📚 Documents Retrieved: {perf['num_docs_retrieved']}")
print(f"   🔍 Retrieval Time: {perf['retrieval_time']:.3f}s")
print(f"   🤖 Generation Time: {perf['generation_time']:.3f}s")
print(f"   ⏱️  Total Time: {perf['total_time']:.3f}s")
print(f"\n💬 Answer:\n{perf['answer']}")

print(f"\n{'='*80}")
print("\n💡 Key Insight: 2-Step RAG has predictable latency!")
print("   - Always 1 retrieval call")
print("   - Always 1 LLM call")
print("   - Total time = retrieval + generation (deterministic)")

⚡ 2-Step RAG Performance Metrics

❓ Question: What are Nike's digital transformation initiatives and e-commerce strategy?

📊 Performance:
   📚 Documents Retrieved: 3
   🔍 Retrieval Time: 2.644s
   🤖 Generation Time: 0.672s
   ⏱️  Total Time: 3.316s

💬 Answer:
The provided context does not explicitly detail Nike's digital transformation initiatives and e-commerce strategy. The text mentions that Nike has NIKE Direct operations, which include both NIKE-owned retail stores and digital platforms (referred to as "NIKE Brand Digital"). It also notes that these digital operations compete with multi-brand retailers and digital commerce platforms.

While it doesn't go into specific details about digital transformation initiatives and e-commerce strategy, we can infer that Nike is likely focused on enhancing its digital presence and consumer engagement through its digital platforms. However, for a more detailed response, I would need additional information from the full 10-K report.


💡 Key Insi

## 🎉 Summary

You've successfully implemented a **2-Step RAG system** using Nike's 10-K annual report!

### What we built:

- ✅ **Basic RAG Chain**: Retrieval → Generation pipeline for Nike financial data
- ✅ **Comparison**: RAG vs non-RAG responses on Nike questions
- ✅ **Interactive Q&A**: Multi-question testing with Nike-specific queries
- ✅ **Source Citations**: Enhanced answers with page number references
- ✅ **Performance Analysis**: Measured latency components

### Key Characteristics of 2-Step RAG:

| Aspect | Description |
|--------|-------------|
| **Predictability** | Always follows: retrieve → generate |
| **Latency** | Fast and deterministic (1 retrieval + 1 LLM call) |
| **Control** | High - you know exactly what happens |
| **Limitations** | Always retrieves, even if not needed |

### When to use 2-Step RAG:

✅ **Good for**:
- Financial document Q&A (like our Nike 10-K example)
- Documentation search
- FAQ systems
- Knowledge base search where context is always needed

❌ **Not ideal for**:
- Multi-step reasoning requiring multiple retrievals
- Questions that don't need retrieval
- Complex queries requiring decision logic

### Real-world Applications:

- **Financial Analysis**: Query annual reports, earnings calls, SEC filings
- **Legal Research**: Search contracts, case law, regulations
- **Technical Documentation**: Developer docs, API references, manuals
- **Customer Support**: Knowledge base search for support articles

### Next Steps:

In **Notebook 03**, we'll implement **Agentic RAG**:
- LLM agent decides when to retrieve from Nike knowledge base
- Tool-based retrieval with decision logic
- Multi-step reasoning for complex Nike business questions
- More flexible but less predictable

## 💡 Additional Resources

- [LangChain RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/)
- [RAG Chains Documentation](https://python.langchain.com/docs/how_to/qa_chat_history_how_to/)
- [Prompt Engineering for RAG](https://www.promptingguide.ai/techniques/rag)
- [Vector Store Retrieval Strategies](https://python.langchain.com/docs/how_to/vectorstores/)